# 1. Data collection

**Source.** CORDIS, the European Commission's public database of EU-funded research projects. We use the official bulk exports released monthly by the Publications Office of the EU on the [EU Open Data Portal](https://data.europa.eu).

**Datasets.** The archives were downloaded manually from the *Distributions* section of each dataset page and stored in `data/raw/`:

| file in `data/raw/` | dataset | role in the project |
|---|---|---|
| `horizon_europe.zip`, `horizon_europe_json.zip` | [Horizon Europe (2021–2027)](https://data.europa.eu/data/datasets/cordis-eu-research-projects-under-horizon-europe-2021-2027) — *HORIZON Projects*, CSV and JSON | primary source for network construction |
| `h2020.zip`, `h2020_json.zip` | [Horizon 2020 (2014–2020)](https://data.europa.eu/data/datasets/cordish2020projects) — *H2020 Projects*, CSV and JSON | historical baseline for the open question |
| `reference/` | [CORDIS reference data](https://data.europa.eu/data/datasets/cordisref-data) — countries, organisation activity types, funding schemes, HORIZON and H2020 programmes | lookup tables to decode codes into readable labels |

**Why two formats.** In the CSV export, the project table (`project.csv`) is malformed: quotes inside the abstracts are not escaped, so about 1% of the rows cannot be parsed reliably. The JSON export does not have this problem, so the project table is rebuilt from it. The participation table (`organization.csv`) is read from the CSV export without issues.

**Tables used.** `project_json.csv`, built in this notebook from the JSON export, where each row describes one funded project (dates, budget, funding scheme, research fields, abstract), and `organization.csv`, where each row records the participation of one organisation in one project (country, activity type, role in the consortium). These are the inputs for the network construction in `02_build_network.ipynb`.

**Output.** This notebook extracts the archives into `data/csv/`, builds the project table from the JSON files and checks that the main tables load correctly. The folders `data/raw/` and `data/csv/` are excluded from the repository (see `.gitignore`) because of their size: to reproduce them, download the archives from the dataset pages listed above into `data/raw/` and run this notebook.

**Snapshot date.** 28 September 2026 (dump version reported on the dataset pages). Dumps are refreshed monthly, so later downloads may yield slightly different counts.


## Imports and configuration

In [1]:
import json
import zipfile
from pathlib import Path

import pandas as pd

# Path to the downloaded zip archives
RAW_DIR = Path("../data/raw")
# Path to the tables extracted from the archives
CSV_DIR = Path("../data/csv")

## Building the tables

Two tables are needed for the network: the **participations** (which organisation took part in which project) and the **projects** (their attributes). Because of the parsing problem described above, they come from two different exports: participations from the CSV archives, projects from the JSON archives.

### Participations and reference data (CSV export)

The two CSV archives are extracted into separate folders, `data/csv/horizon_europe/` and `data/csv/h2020/`. The reference data are handled in the same way: the two zipped programme tables are extracted, while the plain CSV tables are copied as they are, so that all tables end up in `data/csv/reference/`.

In [2]:
# Extract the Horizon Europe and H2020 archives, each into its own folder in data/csv/
for name in ["horizon_europe", "h2020"]:

    zip_path = RAW_DIR / f"{name}.zip"   # archive to extract
    out_folder = CSV_DIR / name          # destination folder

    # Create the destination folder if it does not exist yet
    out_folder.mkdir(parents=True, exist_ok=True)

    # Open the archive and extract all its files into the destination folder
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(out_folder)
        print(f"{zip_path.name}: {len(archive.namelist())} files extracted to {out_folder}")

horizon_europe.zip: 9 files extracted to ..\data\csv\horizon_europe
h2020.zip: 8 files extracted to ..\data\csv\h2020


In [3]:
# Extract the zipped reference tables and copy the plain CSV ones, all into data/csv/reference/
ref_raw = RAW_DIR / "reference"   # downloaded reference files to process
ref_csv = CSV_DIR / "reference"   # destination folder

# Create the destination folder if it does not exist yet
ref_csv.mkdir(parents=True, exist_ok=True)

for file in ref_raw.iterdir():

    # Open the archive and extract all its files into the destination folder
    if file.suffix == ".zip":
        with zipfile.ZipFile(file) as archive:
            archive.extractall(ref_csv)
            print(f"{file.name}: extracted")

    # Copy the CSV file as it is into the destination folder
    elif file.suffix == ".csv":
        (ref_csv / file.name).write_bytes(file.read_bytes())
        print(f"{file.name}: copied")

cordisref-countries.csv: copied
cordisref-H2020programmes-csv.zip: extracted
cordisref-HORIZONprogrammes-csv.zip: extracted
cordisref-organizationActivityType.csv: copied
cordisref-projectFundingSchemeCategory.csv: copied


### Projects (JSON export)

In the JSON export every project is a separate file, containing the project fields at the top level (identifier, title, dates, budget, keywords, abstract) and a list of classifications under `relations.categories`, from which we take the funding scheme, the EuroSciVoc research fields and, for Horizon Europe only, the policy priority markers (relevance for EU priorities such as *Artificial intelligence* or *Climate action*, on a 0–100 scale).

The cell below reads every file directly from the archive and saves the selected fields as one table per programme, `data/csv/<programme>/project_json.csv`. The original `project.csv` is left untouched for comparison.

In [4]:
# Fields to keep from each project JSON file
FIELDS = ["id", "acronym", "title", "status", "startDate", "endDate", "duration",
          "totalCost", "ecMaxContribution", "ecSignatureDate", "keywords", "objective"]

for name in ["horizon_europe", "h2020"]:

    rows = []
    with zipfile.ZipFile(RAW_DIR / f"{name}_json.zip") as archive:
        for filename in archive.namelist():

            # Skip anything that is not a project file
            if not filename.endswith(".json"):
                continue
            with archive.open(filename) as f:
                project = json.loads(f.read().decode("utf-8", errors="replace"))
            
            # Row of the project: value of each field
            row = {field: project.get(field) for field in FIELDS}

            # Add as columns the funding scheme, research fields and policy markers, read from relations.categories
            fields_of_science = []
            for cat in project["relations"].get("categories", []):
                kind = cat["attributes"].get("classification")
                if kind == "projectFundingSchemeCategory":
                    row["fundingScheme"] = cat["code"].strip("/")
                elif kind == "euroSciVoc":
                    fields_of_science.append(cat["title"])
                elif kind == "policyPriorities":
                    row["policy_" + cat["code"].strip("/")] = int(cat["attributes"]["weight"])
            row["euroSciVoc"] = "|".join(fields_of_science)

            rows.append(row)

    # Save the table, one per programme
    projects = pd.DataFrame(rows)
    projects.to_csv(CSV_DIR / name / "project_json.csv", index=False)
    print(f"{name}: {len(projects)} projects saved to project_json.csv")

horizon_europe: 23613 projects saved to project_json.csv
h2020: 35389 projects saved to project_json.csv


## Sanity check

Before building the network, we load the two main Horizon Europe tables and check that they are read correctly: number of rows, number of distinct organisations (the future nodes) and available columns. Note that CORDIS CSV files use `;` as separator.

In [5]:
# Folder containing the Horizon Europe tables
he_folder = CSV_DIR / "horizon_europe"

# Load the two main tables into pandas DataFrames
projects = pd.read_csv(he_folder / "project_json.csv", low_memory=False)
participations = pd.read_csv(he_folder / "organization.csv", sep=";", low_memory=False)

# Basic counts
print("projects:               ", len(projects))
print("participations:         ", len(participations))
print("distinct organisations: ", participations["organisationID"].nunique())

projects:                23613
participations:          145909
distinct organisations:  35200


In [6]:
# Columns available in the two tables, to identify the attributes useful for the analysis
print("project_json.csv columns:")
print(list(projects.columns), "\n")

print("organization.csv columns:")
print(list(participations.columns))

project_json.csv columns:
['id', 'acronym', 'title', 'status', 'startDate', 'endDate', 'duration', 'totalCost', 'ecMaxContribution', 'ecSignatureDate', 'keywords', 'objective', 'policy_digitalAgenda', 'policy_cleanAir', 'policy_ai', 'policy_climate', 'policy_biodiversity', 'fundingScheme', 'euroSciVoc'] 

organization.csv columns:
['projectID', 'projectAcronym', 'organisationID', 'vatNumber', 'name', 'shortName', 'SME', 'activityType', 'street', 'postCode', 'city', 'country', 'nutsCode', 'geolocation', 'organizationURL', 'contactForm', 'contentUpdateDate', 'rcn', 'order', 'role', 'ecContribution', 'netEcContribution', 'totalCost', 'endOfParticipation', 'active']
